In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from catboost import CatBoostRegressor
from typing import Dict, Any, Optional, List, Tuple
import warnings 
warnings.filterwarnings("ignore")

%matplotlib inline 

## Table of Contents
### Feature Description
### Baseline Model
### Baseline Model lag features
### Conclusions

# Feature Description

In [8]:
data = pd.read_csv('/Users/inji/mllearn/data/01_raw/bike_sharing_hour_data.csv')
data.head(3)


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32


In [14]:
data.dtypes

instant         int64
dteday            str
season          int64
yr              int64
mnth            int64
hr              int64
holiday         int64
weekday         int64
workingday      int64
weathersit      int64
temp          float64
atemp         float64
hum           float64
windspeed     float64
casual          int64
registered      int64
cnt             int64
dtype: object

## Step 1 — Prepare the table (no lags yet)

Load **renamed** data (Kedro output), build a datetime, sort by time, drop leakage columns.

- `casual_users` + `registered_users` = `total_users` → do not use them as features
- `instant` is just a row id
- We parse `dteday` + `hr` so a later time split is valid

In [ ]:
# 1. Load the Kedro-renamed table (not the raw file)
df = pd.read_csv("../data/02_intermediate/renamed_data.csv")
df.dtypes

instant                   int64
dteday                      str
season                    int64
yr                        int64
mnth                      int64
hr                        int64
is_holiday                int64
is_weekday                int64
is_workingday             int64
weather                   int64
temperature             float64
apparent_temperature    float64
humidity                float64
windspeed               float64
casual_users              int64
registered_users          int64
total_users               int64
dtype: object

In [26]:
# 1. Load the Kedro-renamed table (not the raw file)
df = pd.read_csv("../data/02_intermediate/renamed_data.csv")

# 2. dteday is a string + hr is 0–23. Combine into a real timestamp.
df["dteday"] = pd.to_datetime(df["dteday"])
df["datetime"] = df["dteday"] + pd.to_timedelta(df["hr"], unit="h")

# 3. Time order matters for any later split (even without lags).
df = df.sort_values("datetime").reset_index(drop=True)

# 4. Leakage / IDs: never put these in X.
LEAKAGE_OR_ID = ["instant", "casual_users", "registered_users"]
df_model = df.drop(columns=LEAKAGE_OR_ID)


# 5. Features vs target. datetime is for splitting/plotting, not a model feature.
TARGET = "total_users"
feature_cols = [
    c
    for c in df_model.columns
    if c not in {TARGET, "datetime", "dteday"}
]
X = df_model[feature_cols]
y = df_model[TARGET]

print(df["datetime"].min(), "→", df["datetime"].max())
print("rows:", len(df_model))
print("features:", feature_cols)
print(
    "check leakage (should be 0):",
    (df["casual_users"] + df["registered_users"] - df["total_users"]).abs().sum(),
)
X.head(3)


2011-01-01 00:00:00 → 2012-12-31 23:00:00
rows: 17379
features: ['season', 'yr', 'mnth', 'hr', 'is_holiday', 'is_weekday', 'is_workingday', 'weather', 'temperature', 'apparent_temperature', 'humidity', 'windspeed']
check leakage (should be 0): 0


,season,yr,mnth,hr,is_holiday,is_weekday,is_workingday,weather,temperature,apparent_temperature,humidity,windspeed
0,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0
1,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0
2,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0


## Step 2 — Time-based train/test split

Rows are already sorted by time (Step 1): oldest at the top, newest at the bottom.

Count how many rows are **before** `2012-10-01`. Take that many from the **start** as train; the rest as test.

`iloc[:n]` means “first n rows”. `iloc[n:]` means “from row n to the end”.

In [29]:
df_model.groupby("dteday")['dteday'].value_counts()

dteday
2011-01-01    24
2011-01-02    23
2011-01-03    22
2011-01-04    23
2011-01-05    23
              ..
2012-12-27    24
2012-12-28    24
2012-12-29    24
2012-12-30    24
2012-12-31    24
Name: count, Length: 731, dtype: int64

In [30]:
# Cutoff is a timestamp: train = strictly before this, test = on/after this.
CUTOFF = pd.Timestamp("2012-10-01")

# Boolean masks aligned with X/y (same row order as df_model).
train_mask = df_model["datetime"] < CUTOFF
test_mask = df_model["datetime"] >= CUTOFF


In [35]:
# Rows are already in time order (Step 1).
CUTOFF = pd.Timestamp("2012-10-01")
n_train = (df_model["datetime"] < CUTOFF).sum()  # how many rows are in the past
print("number of rows :", n_train)


number of rows : 15211


In [36]:
# Rows are already in time order (Step 1).
CUTOFF = pd.Timestamp("2012-10-01")
n_train = (df_model["datetime"] < CUTOFF).sum()  # how many rows are in the past

# First n_train rows = train; remaining rows = test
X_train = X.iloc[:n_train]
y_train = y.iloc[:n_train]
X_test = X.iloc[n_train:]
y_test = y.iloc[n_train:]

print("n_train:", n_train, "n_test:", len(X_test))
print("last train time:", df_model["datetime"].iloc[n_train - 1])
print("first test time:", df_model["datetime"].iloc[n_train])

n_train: 15211 n_test: 2168
last train time: 2012-09-30 23:00:00
first test time: 2012-10-01 00:00:00


In [38]:
X_test.head(3)


,season,yr,mnth,hr,is_holiday,is_weekday,is_workingday,weather,temperature,apparent_temperature,humidity,windspeed
15211,4,1,10,0,0,1,1,1,0.46,0.4545,0.72,0.1045
15212,4,1,10,1,0,1,1,1,0.44,0.4394,0.77,0.0896
15213,4,1,10,2,0,1,1,1,0.46,0.4545,0.72,0.0000
